# Batching and Performance

## What you'll learn

- Understand the two-level batching model (artifacts → execution units → workers)
- Configure `BatchStrategy` fields for different workload patterns
- Measure the impact of batching on pipeline performance
- Choose batching parameters based on workload characteristics

**Prerequisites:** [First Pipeline](../01-getting-started/01-first-pipeline.ipynb).  
**Estimated time:** 20 minutes  
**GPU required:** No.

---

By default, each artifact gets its own execution unit. This tutorial
shows how to group work using `artifacts_per_unit`, explains the full
two-level batching model, and develops intuition for tuning
`BatchStrategy` fields.

In [ ]:
from __future__ import annotations

from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager, Runner
from artisan.utils import tutorial_setup
from artisan.visualization import build_micro_graph, inspect_pipeline

In [ ]:
env = tutorial_setup("batching_performance")

## The two levels of batching

`artifacts_per_unit` groups inputs into **execution units**. A unit shares setup,
postprocessing, an execution record, and the execution-level cache and failure
boundary. By default, its execute phase still calls the operation once per
artifact (or matched multi-input group). Grouping four artifacts into one unit
does not make those four calls one call.

`units_per_worker` packs units into **worker tasks** that process them sequentially.
`max_workers` limits concurrent tasks. The local runner reuses a process pool;
a task does not necessarily start a new process.

An operation that explicitly sets `per_artifact_dispatch=False` receives the
whole unit in one execute call. See
[Writing Creator Operations](../../how-to-guides/writing-creator-operations.md)
for that authoring contract.

## Choose what to group

The two most important fields are `artifacts_per_unit` (how many artifacts
go into each execution unit) and `units_per_worker` (how many units each
worker processes sequentially). For the full list of BatchStrategy fields,
see [Configuring Execution](../../how-to-guides/configuring-execution.md).

Pass these as a dict to the `batch_strategy` parameter of `pipeline.run()`.

## Baseline: default batching

First, run a pipeline with the default configuration (1 artifact per unit)
to establish a timing baseline.

In [ ]:
pipeline = PipelineManager.create(
    name="baseline",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 4, "seed": 42},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
    step_runner=Runner.LOCAL,
)

pipeline.finalize()
inspect_pipeline(env.delta_root)

In [ ]:
build_micro_graph(env.delta_root)

With default batching, the transform step creates 4 execution units — one
per artifact. Each grey box in the micro graph is a separate execution
record. Every artifact gets its own execute call in this example.

## Tuning `artifacts_per_unit`

Group multiple artifacts into fewer execution units to reduce per-unit
overhead. This is the most common tuning parameter.

In [ ]:
env_apu = tutorial_setup("batching_apu")

pipeline = PipelineManager.create(
    name="apu_tuning",
    delta_root=env_apu.delta_root,
    staging_root=env_apu.staging_root,
    working_root=env_apu.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 4, "seed": 42},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    batch_strategy={"artifacts_per_unit": 2},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
    batch_strategy={"artifacts_per_unit": 4},
    step_runner=Runner.LOCAL,
)

pipeline.finalize()
inspect_pipeline(env_apu.delta_root)

In [ ]:
build_micro_graph(env_apu.delta_root)

The transform step now has two execution records, each covering two artifacts.
It still makes four per-artifact execute calls. The metrics step has one
execution record for all four inputs. Grouping reduces repeated unit setup and
recording; it does not by itself combine the computation into one tool call.

## Tuning `units_per_worker`

Packing several units into one task reduces task submissions. A scheduler
provider may also avoid repeated queueing or environment startup. The local
runner submits the packed task to its reusable process pool and runs the units
sequentially there. Larger packs leave fewer tasks available to run in parallel.

In [ ]:
env_upw = tutorial_setup("batching_upw")

pipeline = PipelineManager.create(
    name="upw_tuning",
    delta_root=env_upw.delta_root,
    staging_root=env_upw.staging_root,
    working_root=env_upw.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 4, "seed": 42},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    batch_strategy={"artifacts_per_unit": 2, "units_per_worker": 2},
    step_runner=Runner.LOCAL,
)

pipeline.finalize()
inspect_pipeline(env_upw.delta_root)

In [ ]:
build_micro_graph(env_upw.delta_root)

Four inputs at `artifacts_per_unit=2` produce two units. With
`units_per_worker=2`, one worker task processes both sequentially. The execution
records remain separate because packing changes scheduling, not unit boundaries.

Measure the effect for your runner: packing can reduce scheduling overhead, but
it can also reduce concurrency.

## Operation-level defaults

Operations can declare their own `BatchStrategy` defaults in the class
definition. Step-level overrides take precedence.

```python
class MyExpensiveOperation(OperationDefinition):
    name = "my_expensive_op"

    # Declare default batching for this operation
    batch_strategy = BatchStrategy(
        artifacts_per_unit=10,
        max_workers=4,
    )
```

When you call `pipeline.run(MyExpensiveOperation)` without a `batch_strategy`
override, these defaults are used. If you pass `batch_strategy={"max_workers": 2}`,
the step override wins.

## Measure before tuning

Start with the operation’s defaults and inspect the micro graph. Increase
`artifacts_per_unit` when repeated unit setup or recording dominates. Increase
`units_per_worker` when task scheduling is expensive. Reduce `max_workers` if
concurrent work exceeds available memory or other resources.

Use [Timing Analysis](../08-analysis/04-timing-analysis.ipynb) to compare runs.
These small examples show how work is grouped; their elapsed times are not a
reliable performance benchmark.

## Summary

You grouped four artifacts into fewer execution units, then packed those units
into one worker task. Unit size, task packing, and concurrency control different
costs. Default per-artifact execution still makes a call for each input.

For all available settings, use the `BatchStrategy` docstring through the
[Python API guide](../../reference/python-api.md).

## Next steps

- [Step Overrides](../05-errors-and-control/01-step-overrides.ipynb) — Apply configuration to an individual step
- [Timing Analysis](../08-analysis/04-timing-analysis.ipynb) — Diagnose performance with timing DataFrames
- [Execution Flow](../../concepts/execution-flow.md#batching-and-dispatch) — How two-level batching and dispatch work in the framework
- [Configuring Execution](../../how-to-guides/configuring-execution.md) — Recipes for batching and resource configuration